# Modelado

### 1. Enfoque elegido

Durante la fase de exploración de datos de este proyecto definimos tres preguntas que queríamos responder:

1. ¿Las empresas con mayor inversión en IA por empleado muestran mayores mejoras de productividad, controlando por industria y tamaño empresarial? Para abordarla, planteamos modelar `productivity_change_percent` usando `ai_investment_per_employee` e incorporar al menos `industry` y `company_size` como variables de control.

2.  ¿Las diferencias de nivel de adopción de IA entre industrias se mantienen una vez consideradas características como tamaño empresarial y país? Para abordarla, planteamos comparar industrias controlando por período, tamaño y contexto geográfico, en lugar de atribuir las diferencias únicamente al sector.

3. ¿Una mayor adopción de IA se relaciona con destrucción neta de empleo o con transformación de puestos de trabajo? Para responderla, planteamos construir un balance simple, por ejemplo `net_job_change = jobs_created - jobs_displaced`, y analizar su relación con `ai_adoption_rate`, `task_automation_rate`, industria y tamaño empresarial.

En primer lugar, debemos considerar que las tres variables target (`productivity_change_percent`, `ai_adoption_rate` y `net_job_change`) son contínuas. Esto nos orienta directamente hacia modelos de regresión. 

En segundo lugar, al tener una estructura de datos longitudinal, debemos modelarlo como un **panel**, donde para cada unidad muestral se tienen varias mediciones a lo largo del tiempo. 

Considerando estos puntos, a continuación planteamos un abordaje para cada pregunta:

**H1: Inversión en AI y productividad**

Target: `productivity_change_percent`

Modelo principal: Regresión lineal con efectos mixtos

Predictores:
- `ai_investment_per_employee`
- `industry`
- `company_size`
- `survey_year`
- `quarter`
- `ai_adoption_rate`

Si los predictores están correlacionados, utilizaríamos una regresión regularizada como Ridge o ElasticNet.

Como modelo no lineal para comparación, podríamos utilizar Random Forest o Gradient Boosting. Estos pueden detectar umbrales o efectos no lineales, pero no son tan adecuados como el modelo principal dado que nuestro objetivo es explicar la asociación. 

La fórmula del modelo sería: 
$$
productivity\_change = \beta_0+\beta_1 investment_{it}+\beta_2 industry_{i}+\beta_3 size_i + \beta_4 time_t +\beta_5 adoption_i + u_i + \epsilon_{it}
$$

Aquí, $u_i$ representa el efecto específico de cada compañía. Una regresión con efectos mixtos (GLMM) o una regresión de panel de efectos fijos (modelos TWFE) es preferible a una regresión OLS, dado que las mismas compañías aparecen varias veces. 


**H2: Adopción de IA por industria**

Target: `ai_adoption_rate`

Modelo principal: Regresión lineal con efectos mixtos
- Industria como la principal variable explicativa
- Controles para `company_size`, `country`, `survey_year` y `quarter`
- Efecto de la compañía para compensar las observaciones repetidas. 

Como modelo secundario también se puede ajustar un Gradient Boosting o Random Forest para identificar interacciones no lineales. Por ejemplo, si las diferencias entre industrias son más fuertes para compañías más grandes. 

La pregunta principal que buscamos responder aquí es si los coeficientes de industria se mantienen significativos luego de introducir los controles de tamaño, país y tiempo.


**H3: Transformación del empleo y AI**

En primer lugar, construiremos el target como 
```{python}
company_limpio["net_job_change"] = (
    company_limpio["jobs_created"]
    - company_limpio["jobs_displaced"]
)
```
En esta pregunta tenemos varios abordajes posibles:

**Abordaje de regresión:**

Usando `net_job_change` como el target, se puede aplicar:
- Regresión lineal si la distribución es razonablemente contínua.
- Random Forest o Gradient Boosting si se esperan comportamiento no lineales.
- Regresión Poisson o Binomial Negativa si deseamos modelar `jobs_created` y `jobs_displaced` de manera separada como conteos.

Los predictores más importantes serían:

- `ai_adoption_rate`
- `task_automation_rate`
- `ai_maturity_score`
- `industry`
- `company_size`
- `num_employees`
- `survey_year`
- `quarter`

**Abordaje de clasificación**

Creando una variable target categórica dicotómica
```{python}
company_limpio["employment_effect"] = np.select(
    [
        company_limpio["net_job_change"] > 0,
        company_limpio["net_job_change"] < 0
    ],
    [
        "net_creation",
        "net_displacement"
    ],
    default="neutral"
)
```
Se podría usar:
- Regresión logística multinomial
- Random Forest o Gradient Boosting para una clasificación no lineal
- Evaluación con macro-F1, balanced accuracy y matriz de confusión, dado que las categorías podrían no estar balanceadas. 

Por último, respecto del split train/test, consideramos que la mejor manera es realizar un splitting agrupado por company_id. De esa manera, todas las observaciones de una misma empresa estarán en uno de los conjuntos de datos.

### 2. Preparación final

#### 2.1 Carga del dataset curado completo

Para esta etapa cargamos `data_curada_completa.csv`, que conserva las variables originales y las variables derivadas durante la curación. Esta tabla permite conservar `company_id`, `country`, `industry`, `survey_year` y `quarter` como información estructural del panel.

`company_id` se utilizará para agrupar las observaciones durante la partición entre entrenamiento y prueba, pero no se incorporará como predictor.

In [1]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

RANDOM_STATE = 42

datos = pd.read_csv('../data/data_curada_completa.csv')

columnas_panel = [
    'company_id', 'survey_year', 'quarter',
    'country', 'industry', 'company_size'
]
faltantes_panel = sorted(set(columnas_panel) - set(datos.columns))
assert not faltantes_panel, f'Faltan columnas del panel: {faltantes_panel}'

print(f'Dataset cargado: {datos.shape[0]:,} filas x {datos.shape[1]} columnas')
print('Columnas estructurales disponibles:', columnas_panel)

Dataset cargado: 143,073 filas x 77 columnas
Columnas estructurales disponibles: ['company_id', 'survey_year', 'quarter', 'country', 'industry', 'company_size']


#### 2.2 Construcción del índice temporal

`period_index` representa cada trimestre en una escala entera ordenada. Se conservan `survey_year` y `quarter` para poder modelar sus efectos por separado en las etapas siguientes.

In [2]:
filas_antes_panel = len(datos)
datos = datos.dropna(
    subset=['company_id', 'survey_year', 'quarter', 'response_id']
).copy()
print(
    f"Filas sin estructura temporal o identificador eliminadas: "
    f"{filas_antes_panel - len(datos):,}"
)

datos['quarter_num'] = (
    datos['quarter']
    .astype('string')
    .str.extract(r'(\d+)', expand=False)
    .astype('Int64')
)
assert datos['quarter_num'].between(1, 4).all(), 'quarter inválido'

primer_anio = datos['survey_year'].min()
datos['period_index'] = (
    (datos['survey_year'] - primer_anio) * 4
    + datos['quarter_num'] - 1
).astype(int)

datos = datos.sort_values(
    ['company_id', 'period_index', 'response_id'],
    kind='stable'
).reset_index(drop=True)

print(
    f"Períodos: {datos['period_index'].min()} "
    f"a {datos['period_index'].max()}"
)
display(datos[['period_index', 'survey_year', 'quarter']].drop_duplicates().sort_values('period_index').head())

Filas sin estructura temporal o identificador eliminadas: 1
Períodos: 0 a 15


,period_index,survey_year,quarter
27,0,2023.0,Q1
0,1,2023.0,Q2
14,2,2023.0,Q3
1,3,2023.0,Q4
2,4,2024.0,Q1


#### 2.3 Targets y tasas de empleo

H1 utiliza `productivity_change_percent`, H2 utiliza `ai_adoption_rate` y H3 utiliza `net_job_change`, calculado como puestos creados menos puestos desplazados. Para H3 también se calculan tasas relativas al tamaño de la empresa, porque los conteos absolutos están fuertemente condicionados por `num_employees`.

Los conteos de empleo no se usarán como predictores de H3 cuando formen parte del target o lo midan directamente.

In [3]:
requeridas_h3 = [
    'jobs_created', 'jobs_displaced',
    'reskilled_employees', 'num_employees'
]
faltantes_h3 = sorted(set(requeridas_h3) - set(datos.columns))
assert not faltantes_h3, f'Faltan columnas para H3: {faltantes_h3}'

denominador_empleados = datos['num_employees'].clip(lower=1)
datos['net_job_change'] = datos['jobs_created'] - datos['jobs_displaced']
datos['job_creation_rate'] = datos['jobs_created'] / denominador_empleados
datos['job_displacement_rate'] = datos['jobs_displaced'] / denominador_empleados
datos['reskilling_rate'] = datos['reskilled_employees'] / denominador_empleados
datos['net_job_change_rate'] = datos['net_job_change'] / denominador_empleados

print('Targets y tasas construidos correctamente.')

Targets y tasas construidos correctamente.


#### 2.4 Datasets específicos por hipótesis

Cada hipótesis tiene una matriz de predictores diferente. Se conservan las columnas de identificación y del panel como metadatos, pero `company_id` no entra como predictor.

H1 modela productividad con inversión, adopción, tamaño, escala, contexto del país y tiempo. H2 modela adopción con industria como variable principal y controles de tamaño, país y período. H3 modela el cambio neto de empleo con adopción, automatización, tamaño, contexto y tiempo.

Se excluyen las columnas `avg_*` por ser agregados construidos por industria y se evita usar en H2 variables que son transformaciones muy próximas de la adopción, como `ai_stage_enc` y `ai_maturity_score`.

In [4]:
metadatos_panel = [
    'company_id', 'period_index', 'survey_year',
    'quarter', 'country', 'industry', 'company_size'
]

predictores_h1 = [
    'ai_investment_per_employee', 'industry', 'company_size',
    'survey_year', 'quarter', 'ai_adoption_rate', 'num_employees',
    'annual_revenue_usd_millions', 'company_age', 'gdp_per_capita',
    'digital_maturity_index'
]

predictores_h2 = [
    'industry', 'company_size', 'country', 'survey_year', 'quarter',
    'num_employees', 'annual_revenue_usd_millions', 'gdp_per_capita',
    'digital_maturity_index', 'internet_penetration'
]

predictores_h3 = [
    'ai_adoption_rate', 'task_automation_rate', 'ai_maturity_score',
    'industry', 'company_size', 'country', 'num_employees',
    'annual_revenue_usd_millions', 'ai_projects_active', 'years_using_ai',
    'survey_year', 'quarter', 'gdp_per_capita', 'digital_maturity_index'
]

def construir_dataset_hipotesis(df, target, predictores):
    columnas = metadatos_panel + [target] + [
        columna for columna in predictores
        if columna not in metadatos_panel and columna != target
    ]
    columnas = list(dict.fromkeys(columnas))
    faltantes = sorted(set(columnas) - set(df.columns))
    assert not faltantes, f'Faltan columnas para {target}: {faltantes}'

    dataset = df[columnas].dropna().copy()
    predictores_reales = [
        columna for columna in predictores
        if columna in dataset.columns and columna not in metadatos_panel
    ]
    return (
        dataset,
        dataset[predictores_reales].copy(),
        dataset[target].copy(),
        dataset['company_id'].copy(),
    )

datos_h1, X_h1, y_h1, grupos_h1 = construir_dataset_hipotesis(
    datos, 'productivity_change_percent', predictores_h1
)
datos_h2, X_h2, y_h2, grupos_h2 = construir_dataset_hipotesis(
    datos, 'ai_adoption_rate', predictores_h2
)
datos_h3, X_h3, y_h3, grupos_h3 = construir_dataset_hipotesis(
    datos, 'net_job_change', predictores_h3
)

assert 'net_job_change' not in X_h3.columns
assert not any(col.startswith('avg_') for col in X_h1.columns)
assert not any(col.startswith('avg_') for col in X_h2.columns)

print(f'H1: {datos_h1.shape[0]:,} filas x {X_h1.shape[1]} predictores')
print(f'H2: {datos_h2.shape[0]:,} filas x {X_h2.shape[1]} predictores')
print(f'H3: {datos_h3.shape[0]:,} filas x {X_h3.shape[1]} predictores')

H1: 143,072 filas x 7 predictores
H2: 143,072 filas x 5 predictores
H3: 143,071 filas x 9 predictores


#### 2.5 Partición train/test agrupada por empresa

`GroupShuffleSplit` garantiza que todas las observaciones de una misma empresa queden completamente en entrenamiento o completamente en prueba. Esta partición evalúa la generalización hacia empresas no observadas y evita fuga entre registros repetidos de una misma compañía.

In [5]:
def split_por_empresa(dataset, X, y, grupos, test_size=0.20):
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=test_size,
        random_state=RANDOM_STATE,
    )
    train_idx, test_idx = next(splitter.split(X, y, groups=grupos))

    empresas_train = set(dataset.iloc[train_idx]['company_id'])
    empresas_test = set(dataset.iloc[test_idx]['company_id'])
    assert empresas_train.isdisjoint(empresas_test)

    return {
        'datos_train': dataset.iloc[train_idx].copy(),
        'datos_test': dataset.iloc[test_idx].copy(),
        'X_train': X.iloc[train_idx].copy(),
        'X_test': X.iloc[test_idx].copy(),
        'y_train': y.iloc[train_idx].copy(),
        'y_test': y.iloc[test_idx].copy(),
    }

splits = {
    'H1': split_por_empresa(datos_h1, X_h1, y_h1, grupos_h1),
    'H2': split_por_empresa(datos_h2, X_h2, y_h2, grupos_h2),
    'H3': split_por_empresa(datos_h3, X_h3, y_h3, grupos_h3),
}

for nombre, particion in splits.items():
    train = particion['datos_train']
    test = particion['datos_test']
    print(
        f"{nombre}: train={len(train):,} filas "
        f"({train['company_id'].nunique():,} empresas), "
        f"test={len(test):,} filas "
        f"({test['company_id'].nunique():,} empresas)"
    )

H1: train=114,398 filas (8,014 empresas), test=28,674 filas (2,004 empresas)
H2: train=114,398 filas (8,014 empresas), test=28,674 filas (2,004 empresas)
H3: train=114,394 filas (8,013 empresas), test=28,677 filas (2,004 empresas)


#### 2.6 Codificación de variables categóricas y transformaciones

Los modelos lineales necesitan una representación numérica de las variables categóricas. Se construyen dummies con `drop_first=True` para evitar colinealidad perfecta en los efectos fijos observables. `company_id` no se transforma en dummies: se conserva como identificador para los efectos de empresa y para verificar la estructura del panel.

Para H1 y H2 se aplica `log1p` únicamente a variables continuas no negativas y sesgadas. Las transformaciones se definen antes del ajuste del modelo y se aplican de forma consistente a los conjuntos de entrenamiento y prueba. No se transforma el target ni variables categóricas. 

En H1, el modelo TWFE absorberá los efectos fijos de empresa y período. Por eso `industry` y `company_size`, si son constantes dentro de una empresa, no se estiman como coeficientes independientes en ese modelo, aunque se conservarán en los datos para describir la muestra y para modelos alternativos.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Variables categóricas que se usarán como efectos fijos observables.
categoricas_h1 = ['industry', 'company_size', 'quarter']
categoricas_h2 = ['industry', 'company_size', 'country', 'quarter']
categoricas_h3 = ['industry', 'company_size', 'country', 'quarter']

# Variables continuas no negativas y sesgadas que se transforman para H1 y H2.
log_h1 = [
    'ai_investment_per_employee',
    'num_employees',
    'annual_revenue_usd_millions',
]
log_h2 = [
    'num_employees',
    'annual_revenue_usd_millions',
]

def preparar_predictores(df, predictores, categoricas, variables_log=None):
    """Aplica log1p y dummies sin modificar el dataset original."""
    variables_log = variables_log or []
    resultado = df[predictores].copy()

    for columna in variables_log:
        if columna in resultado.columns:
            if (resultado[columna] < 0).any():
                raise ValueError(f'{columna} contiene valores negativos para log1p.')
            resultado[f'{columna}_log1p'] = np.log1p(resultado[columna])
            resultado = resultado.drop(columns=columna)

    categoricas_presentes = [
        columna for columna in categoricas
        if columna in resultado.columns
    ]
    resultado = pd.get_dummies(
        resultado,
        columns=categoricas_presentes,
        drop_first=True,
        dtype=float,
    )
    return resultado.astype(float)

X_h1_dummy = preparar_predictores(
    datos_h1, predictores_h1, categoricas_h1, variables_log=log_h1
)
X_h2_dummy = preparar_predictores(
    datos_h2, predictores_h2, categoricas_h2, variables_log=log_h2
)
X_h3_dummy = preparar_predictores(
    datos_h3, predictores_h3, categoricas_h3
)

print(f'H1 con dummies y log1p: {X_h1_dummy.shape}')
print(f'H2 con dummies y log1p: {X_h2_dummy.shape}')
print(f'H3 con dummies: {X_h3_dummy.shape}')

H1 con dummies y log1p: (143072, 21)
H2 con dummies y log1p: (143072, 48)
H3 con dummies: (143071, 52)


### 3. Implementación y evaluación

### 4. Interpretación de resultados

### 5. Vuelta a las preguntas del P1

### 6. Conclusiones finales